In [2]:

# =============================================================================
# BENCHMARK FINNN vs GNN su QM9 (versione corretta)
# =============================================================================

# 1. Installazione dipendenze
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install -q torch-geometric
!pip install -q rdkit

# 2. Download moduli Novae dal repository
!mkdir -p novabene finnn
!wget -q -O novabene/arithmetic.py https://raw.githubusercontent.com/oppomariolevi-ai/Novae/main/novabene/arithmetic.py
!wget -q -O novabene/graph.py https://raw.githubusercontent.com/oppomariolevi-ai/Novae/main/novabene/graph.py
!wget -q -O finnn/neuron.py https://raw.githubusercontent.com/oppomariolevi-ai/Novae/main/finnn/neuron.py

import sys
sys.path.insert(0, '/content')

# 3. Import librerie
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import numpy as np
import time

# 4. Caricamento dataset QM9 (5000 molecole per test rapido)
dataset = QM9(root='data/QM9').shuffle()[:5000]
print(f"Dataset caricato: {len(dataset)} molecole")

# Split train/val/test
train_dataset = dataset[:3500]
val_dataset = dataset[3500:4250]
test_dataset = dataset[4250:]

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

# 5. Modello GNN standard (baseline)
class GNNModel(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(dataset.num_node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)  # target: proprietà 0 (dipolo)
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.fc(x).squeeze()

# 6. Modello FINNN
from finnn.neuron import FINNNLayer

class FINNNModel(nn.Module):
    def __init__(self, hidden_dim=52):
        super().__init__()
        self.embed = nn.Linear(dataset.num_node_features, 52)
        self.finnn = FINNNLayer(lambda_val=0.5)
        self.fc = nn.Linear(52, 1)
    def forward(self, data):
        x, batch = data.x, data.batch
        x = F.relu(self.embed(x))
        x = self.finnn(x)
        x = global_mean_pool(x, batch)
        return self.fc(x).squeeze()

# 7. Funzioni di training e valutazione
def train(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = F.mse_loss(out, data.y[:, 0])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    total_mae = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)
            mae = F.l1_loss(out, data.y[:, 0], reduction='sum').item()
            total_mae += mae
    return total_mae / len(loader.dataset)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

# 8. Addestramento e valutazione
gnn = GNNModel(hidden_dim=64).to(device)
optimizer_gnn = torch.optim.Adam(gnn.parameters(), lr=0.001)
print(f"GNN parametri: {sum(p.numel() for p in gnn.parameters())}")

finnn = FINNNModel(hidden_dim=52).to(device)
optimizer_finnn = torch.optim.Adam(finnn.parameters(), lr=0.001)
print(f"FINNN parametri: {sum(p.numel() for p in finnn.parameters())}")

epochs = 10
print("\n--- Addestramento GNN ---")
start = time.time()
for epoch in range(1, epochs+1):
    loss = train(gnn, train_loader, optimizer_gnn, device)
    val_mae = evaluate(gnn, val_loader, device)
    print(f"Epoca {epoch:2d}: Loss={loss:.4f}, Val MAE={val_mae:.4f}")
gnn_time = time.time() - start

print("\n--- Addestramento FINNN ---")
start = time.time()
for epoch in range(1, epochs+1):
    loss = train(finnn, train_loader, optimizer_finnn, device)
    val_mae = evaluate(finnn, val_loader, device)
    print(f"Epoca {epoch:2d}: Loss={loss:.4f}, Val MAE={val_mae:.4f}")
finnn_time = time.time() - start

gnn_test_mae = evaluate(gnn, test_loader, device)
finnn_test_mae = evaluate(finnn, test_loader, device)

print("\n=== RISULTATI FINALI ===")
print(f"GNN   - Parametri: {sum(p.numel() for p in gnn.parameters())}, Test MAE: {gnn_test_mae:.4f}, Tempo: {gnn_time:.1f}s")
print(f"FINNN - Parametri: {sum(p.numel() for p in finnn.parameters())}, Test MAE: {finnn_test_mae:.4f}, Tempo: {finnn_time:.1f}s")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 20.0 MB/s eta 0:00:00


Extracting data/QM9/raw/qm9.zip


Dataset caricato: 5000 molecole
Dispositivo: cpu
GNN parametri: 9153
FINNN parametri: 3433

--- Addestramento GNN ---
Epoca  1: Loss=3.6873, Val MAE=1.0336
Epoca  2: Loss=2.1297, Val MAE=1.0355
Epoca  3: Loss=2.0920, Val MAE=1.0247
Epoca  4: Loss=2.0697, Val MAE=1.0289
Epoca  5: Loss=2.0522, Val MAE=1.0219
Epoca  6: Loss=2.0270, Val MAE=1.0010
Epoca  7: Loss=2.0068, Val MAE=1.0066
Epoca  8: Loss=1.9922, Val MAE=0.9920
Epoca  9: Loss=1.9852, Val MAE=0.9743
Epoca 10: Loss=1.9642, Val MAE=0.9779

--- Addestramento FINNN ---
Epoca  1: Loss=3.5356, Val MAE=1.0641
Epoca  2: Loss=2.1013, Val MAE=1.0437
Epoca  3: Loss=2.0856, Val MAE=1.0283
Epoca  4: Loss=2.0691, Val MAE=1.0222
Epoca  5: Loss=2.0568, Val MAE=1.0243
Epoca  6: Loss=2.0447, Val MAE=1.0073
Epoca  7: Loss=2.0251, Val MAE=1.0006
Epoca  8: Loss=2.0155, Val MAE=0.9875
Epoca  9: Loss=1.9990, Val MAE=0.9815
Epoca 10: Loss=1.9868, Val MAE=0.9762

=== RISULTATI FINALI ===
GNN   - Parametri: 9153, Test MAE: 1.0778, Tempo: 11.7s
FINNN - Par